# Lab: retracking radar altimeter waveforms

```{note}
This lab accompanies {doc}`radar-altimetry`. The electromagnetic reason microwaves enter dry snow is in {doc}`../radar/em-waves`, and the mass-balance bookkeeping that makes elevation change worth measuring is in {doc}`../ice_flow/mass-balance`.
```

A radar altimeter does not measure height. It measures time, the round-trip delay of a microwave pulse that reflects from whatever is below. Over ice sheets that reflection is complicated because dry firn is not a sharp surface — it is a graded dielectric that backscatters energy from a range of depths, spreading the return in time. The process of identifying where the ice surface sits inside that spread is called *retracking*, and it is the central algorithm standing between the raw Level-1b waveform and the Level-2 height product that eventually becomes an elevation-change time series.

By the end of this lab you should be able to

- describe what a CryoSat-2 L1b file actually contains and why it is far larger than its L2 companion,
- load a pass of waveforms from a netCDF archive and visualise their along-track evolution,
- implement the threshold-of-peak and OCOG retrackers from scratch,
- convert retracked range to surface height using the geophysical corrections already stored in the file,
- compare the two retrackers and identify regions where they diverge, and
- connect retracker consistency to the integrity of the multi-decadal elevation-change record {cite}`wingham2006`.

## 1. Where the data live, and how much of it there is

CryoSat-2 L1b products are distributed through ESA's CryoSat Science Server at `https://science-pds.cryosat.esa.int`. You register once, then download via the SciHub-style portal or the `cs2tools` Python package. The analogous product for Sentinel-3A and -3B lives on the Copernicus Data Space at `https://dataspace.copernicus.eu`; it carries the suffix `SR_1_SRA____` for the SAR-mode L1b and `SR_2_LAN____` for the L2 land-ice heights.

### Data volume — read this before you download anything

A single CryoSat-2 orbit of L1b data is roughly **700 MB** compressed. That is because every 20-Hz measurement record carries the full 128-sample waveform (or 512 samples in SAR mode), the Doppler-stack of unfocused looks before summation, the instrument internal calibration tables, and a dense set of geolocation fields. Multiply by the roughly 14 orbits per day and you reach about 10 GB per day — and the mission has been running since 2010. The matching L2 product, which stores only the retracked range, the geophysical corrections, and the estimated height at each 20-Hz shot, is around **15 MB per orbit**, roughly fifty times smaller. Most science questions — basin-scale volume change, seasonal cycles, secular trend — can be answered entirely from L2.

So when should you download L1b and retrack yourself?

**Near margins and over sloping outlet glaciers**, the standard ESA retracker (a threshold-of-peak variant tuned for interior ice sheets) tends to lock onto off-nadir returns from nearby valley walls rather than the true surface. A custom retracker, or a retracker that also uses the CryoSat-2 interferometric SARIn mode angle measurement, can recover height where the standard product fails or is simply absent. **Over regions with seasonal wetness**, a retracker that explicitly accounts for the penetrating power of dry versus wet snow performs better than one tuned to the mean annual state. **When you suspect retracker drift** — a systematic change in which part of the waveform is called the surface — you need the L1b to apply a consistent algorithm across the full time series. For this lab, **download one orbit pass over a single drainage basin**, not a mission archive.

For a fast start without registration, ESA provides a small test granule in the CryoSat software tools distribution; its path is noted in the `# verify:` comment in the load cell below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# netCDF4 is the standard reader for CryoSat-2 and Sentinel-3 L1b products.
# Install with: pip install netCDF4
import netCDF4 as nc

# Adjust this to wherever you placed the downloaded L1b granule.
# CryoSat-2 SAR-mode L1b files follow the naming pattern:
#   CS_OFFL_SIR_SAR_1B_<starttime>_<endtime>_<baseline>.nc
# The test granule shipped with cs2tools is usually at:
#   ~/cs2tools/test_data/CS_OFFL_SIR_SAR_1B_20190101T*.nc
L1B_PATH = Path('data/CS_OFFL_SIR_SAR_1B_20190101T120000_20190101T121500_D001.nc')  # EDIT ME

## 2. Loading a pass: what is actually inside the file

A CryoSat-2 L1b granule in netCDF format organises its content into a small number of groups. The `data_20` group holds the 20-Hz measurements; within it, `ku` is the Ku-band sub-group carrying the waveforms (CryoSat-2 has Ku and S bands, but S-band is rarely used over ice). The waveform array has dimensions `(n_records, n_bins)` where `n_records` is the number of 20-Hz shots and `n_bins` is 128 in LRM mode or 512 in SAR mode. Alongside the waveforms you will find

- `window_delay` — the time (seconds) from pulse transmission to the start of the receive window, which anchors the first waveform bin in range,
- `range_cor_ice_oce` (or a similarly named field) — the sum of all geophysical and atmospheric range corrections precomputed by ESA: dry troposphere, wet troposphere, ionosphere, ocean loading tide, solid Earth tide, pole tide, and the across-track slope correction,
- `lat_20_ku`, `lon_20_ku` — geodetic coordinates of the sub-satellite point at each shot, and
- `alt_20_ku` — the satellite altitude above the reference ellipsoid.

The height formula is simply

$$h = h_\text{sat} - R_\text{retracked} - \Delta R_\text{corrections},$$

where $R_\text{retracked}$ is the range you extract from the waveform (in metres, converted from the time delay of the chosen bin at the speed of light in vacuum), and $\Delta R_\text{corrections}$ is the sum of all the correction fields, which are stored as signed quantities that *add* to the range in the direction that brings the apparent surface closer to reality. In the cell below, `# verify:` flags mark every group and variable name that you should check against the actual file with `ncdump -h` or `ds.groups` before running.

In [ ]:
# Open the L1b file and extract the arrays needed for the rest of the lab.
# All variable names are marked # verify: — check them against your file
# with:  ncdump -h <your_file>.nc

with nc.Dataset(L1B_PATH, 'r') as ds:
    grp = ds['data_20']['ku']            # verify: group path 'data_20/ku'

    waveforms  = grp['pwr_waveform_20_ku'][:].astype(np.float64)   # verify: var name
    win_delay  = grp['window_delay_20_ku'][:]                       # verify: var name, seconds
    alt_sat    = grp['alt_20_ku'][:]                                # verify: satellite altitude, m
    lat        = grp['lat_20_ku'][:]                                # verify: degrees north
    lon        = grp['lon_20_ku'][:]                                # verify: degrees east
    range_cor  = grp['range_cor_ice_oce_20_ku'][:]                  # verify: total correction, m

n_records, n_bins = waveforms.shape
C_LIGHT = 299_792_458.0   # m/s, speed of light in vacuum

# Bandwidth determines the bin spacing in range. CryoSat-2 SAR uses 320 MHz.
BANDWIDTH = 320e6   # Hz  # verify: check mission/mode documentation
BIN_RANGE = C_LIGHT / (2.0 * BANDWIDTH)   # metres per range bin

# Range from satellite to the first bin, converted from time delay.
range_window_start = win_delay * C_LIGHT / 2.0   # metres

# Build a range array: shape (n_records, n_bins)
bins = np.arange(n_bins)
range_matrix = range_window_start[:, np.newaxis] + bins[np.newaxis, :] * BIN_RANGE

print(f'Records: {n_records},  Bins per waveform: {n_bins}')
print(f'Bin spacing in range: {BIN_RANGE*100:.2f} cm')
print(f'Along-track extent: {abs(lat[-1]-lat[0]):.2f} degrees ({abs(lat[-1]-lat[0])*111:.0f} km)')

## 3. Visualising the waveform stack

Before writing a single retracker, plot the waveforms as a two-dimensional image — range bin on the horizontal axis, shot number (equivalently, along-track distance) on the vertical axis, and normalised power as colour. This *waveform stack* is the closest thing to a raw radar image the altimeter produces.

What to look for

**Leading-edge position:** the bright band where waveforms first light up is the leading edge, which traces the surface. Its horizontal position in the bin array shifts as the surface topography changes; a slope of a few degrees can push it from one end of the receive window to the other.

**Peakiness:** over smooth interior ice the waveforms are narrow and bright at the peak, then trail off. Over rough margins they are broader, flatter, sometimes multi-peaked. Peakiness is a diagnostic of surface roughness within the altimeter footprint.

**Sub-surface scatter over dry firn:** where firn is cold and dry, the microwave penetrates tens of centimetres to a few metres before all energy is absorbed or scattered back {cite}`wingham2006`. This spreads the trailing edge of the waveform, shifting the apparent surface upward into the firn column. If the firn's dielectric character changes between passes — say, a late-summer melt crust refreezes — the apparent height changes with it, even if the true surface did not move. This is the penetration bias discussed in {doc}`radar-altimetry` and rooted in the dielectric theory of {doc}`../radar/em-waves`.

In [ ]:
# Normalise each waveform to its own peak so the leading-edge structure
# is visible regardless of along-track gain variations.
wf_norm = waveforms / np.maximum(waveforms.max(axis=1, keepdims=True), 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# --- panel 1: waveform stack ---
ax = axes[0]
im = ax.imshow(
    wf_norm,
    aspect='auto',
    origin='lower',
    cmap='plasma',
    vmin=0, vmax=1,
    extent=[0, n_bins, 0, n_records]
)
plt.colorbar(im, ax=ax, label='normalised power')
ax.set_xlabel('range bin')
ax.set_ylabel('shot number (along track)')
ax.set_title('Waveform stack — full pass')

# --- panel 2: three example waveforms ---
ax2 = axes[1]
sample_indices = [n_records // 4, n_records // 2, 3 * n_records // 4]
colors = ['steelblue', 'darkorange', 'green']
for idx, c in zip(sample_indices, colors):
    ax2.plot(wf_norm[idx], label=f'shot {idx}  (lat {lat[idx]:.2f}°)', color=c, lw=1.5)

ax2.axvline(n_bins // 2, color='gray', ls=':', lw=1, label='window centre')
ax2.set_xlabel('range bin')
ax2.set_ylabel('normalised power')
ax2.set_title('Individual waveforms')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Two retrackers from scratch

A retracker answers one question: given a waveform vector $W(b)$ where $b$ is the bin index, which bin corresponds to the ice surface? The answer is a fractional bin number $b^*$ that, multiplied by the bin range spacing and added to the window-start range, gives the satellite-to-surface range $R$.

### 4.1 Threshold-of-peak

The simplest and most widely used retracker finds the peak bin, walks back toward the leading edge, and stops at the first bin where power crosses a fraction $\alpha$ of the peak. ESA used a 50%-of-peak threshold for ERS and Envisat; values between 20% and 80% appear in the literature. Linear interpolation between the two bins that bracket the threshold gives the fractional bin.

Its strength is robustness — it works on almost any waveform shape. Its weakness is that the retracked point shifts as the waveform shape changes, so a seasonal softening of the firn that broadens the leading edge moves $b^*$ even at constant surface height. That is the penetration-bias problem in operational form.

### 4.2 OCOG (Offset Centre of Gravity)

The OCOG retracker {cite}`[TODO-CITE: Wingham et al. 1986 Geophys. Res. Lett. OCOG retracker]` does not look for a threshold. It computes the amplitude $A$ and centre-of-gravity $C$ of the waveform using only the signal bins:

$$A = \sqrt{\frac{\sum_b W(b)^4}{\sum_b W(b)^2}}, \qquad C = \frac{\sum_b b\, W(b)^2}{\sum_b W(b)^2}.$$

The retracked range then corresponds to the bin at which a rectangular waveform of amplitude $A$ and total area equal to $\sum_b W(b)^2$ would begin, which works out to

$$b^* = C - \frac{1}{2}\frac{\sum_b W(b)^2}{A}.$$

OCOG is less sensitive to the absolute power level and more stable over waveforms with extended tails. It tends to track a slightly deeper surface than a 50%-threshold retracker over dry firn, so the two retrackers bracket the penetration-depth uncertainty.

In [ ]:
def retrack_threshold(waveform, alpha=0.50):
    """Threshold-of-peak retracker.

    Parameters
    ----------
    waveform : 1-D array, power values
    alpha    : threshold fraction of peak power (default 0.50)

    Returns
    -------
    Fractional bin number of the retracked leading-edge point.
    Returns NaN if the waveform is all zeros or the leading edge
    cannot be found.
    """
    peak_idx = int(np.argmax(waveform))
    threshold = alpha * waveform[peak_idx]
    if threshold <= 0:
        return np.nan
    # Walk from the peak back toward bin 0, looking for the crossing.
    for i in range(peak_idx, 0, -1):
        if waveform[i - 1] < threshold <= waveform[i]:
            # Linear interpolation for the fractional bin.
            frac = (threshold - waveform[i - 1]) / (waveform[i] - waveform[i - 1])
            return (i - 1) + frac
    return float(peak_idx)  # fallback: peak itself


def retrack_ocog(waveform):
    """OCOG (Offset Centre of Gravity) retracker.

    Returns the fractional bin number of the OCOG leading-edge estimate.
    """
    w2 = waveform ** 2
    w4 = waveform ** 4
    denom2 = np.sum(w2)
    if denom2 == 0:
        return np.nan
    A = np.sqrt(np.sum(w4) / denom2)
    if A == 0:
        return np.nan
    bins = np.arange(len(waveform), dtype=np.float64)
    C = np.sum(bins * w2) / denom2
    width = denom2 / A
    b_star = C - 0.5 * width
    return b_star


# Apply both retrackers to every waveform in the pass.
b_thresh = np.array([retrack_threshold(waveforms[i]) for i in range(n_records)])
b_ocog   = np.array([retrack_ocog(waveforms[i])     for i in range(n_records)])

# Convert fractional bin to range (metres).
R_thresh = range_window_start + b_thresh * BIN_RANGE
R_ocog   = range_window_start + b_ocog   * BIN_RANGE

# Height = satellite altitude - retracked range - geophysical corrections.
# range_cor is already a positive quantity that shortens the apparent range
# (sign convention: verify against your file's attribute 'long_name').
h_thresh = alt_sat - R_thresh - range_cor   # verify: sign of range_cor
h_ocog   = alt_sat - R_ocog   - range_cor   # verify: sign of range_cor

print(f'Height statistics (threshold 50%): mean {np.nanmean(h_thresh):.1f} m, '
      f'std {np.nanstd(h_thresh):.1f} m')
print(f'Height statistics (OCOG):          mean {np.nanmean(h_ocog):.1f} m, '
      f'std {np.nanstd(h_ocog):.1f} m')

## 5. Comparing the two retrackers along the pass

Plot the two height profiles against latitude, then plot their difference. The difference is a direct diagnostic of where the waveform shape is doing something unusual.

Over flat interior ice, waveforms are symmetric and both retrackers land within a few centimetres of each other; the ESA L2 product would look exactly like either of them. Where the retrackers diverge — at an outlet glacier, across a facies boundary, or where a wind-scour trough has roughened the surface — the divergence tells you the surface is *interesting* in a way that justifies downloading the L1b and making your own choice.

The magnitude of the divergence also bounds the penetration uncertainty. If dry-firn waveforms push OCOG about one metre deeper than the threshold retracker, and the firn dielectric changes by 10% between repeat passes, you might expect a few centimetres of spurious elevation change in a year-over-year difference — comparable to the geodetic signal in low-accumulation regions.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(lat, h_thresh, lw=1.0, color='steelblue', label='threshold 50%')
ax1.plot(lat, h_ocog,   lw=1.0, color='darkorange', alpha=0.8, label='OCOG')
ax1.set_ylabel('height above ellipsoid (m)')
ax1.set_title('Retracked surface heights along the pass')
ax1.legend()

diff = h_thresh - h_ocog
ax2.plot(lat, diff, lw=1.0, color='firebrick')
ax2.axhline(0.0, color='gray', lw=0.5)
ax2.set_ylabel('threshold − OCOG (m)')
ax2.set_xlabel('latitude (°)')
ax2.set_title('Retracker difference — large values flag unusual surfaces')

plt.tight_layout()
plt.show()

# Mark the five shots with the largest absolute difference.
top5 = np.argsort(np.abs(diff))[-5:][::-1]
print('Shots with largest retracker divergence:')
for i in top5:
    print(f'  shot {i:4d}  lat {lat[i]:7.3f}°  diff {diff[i]:+.3f} m')

## 6. Tasks

### Task 1: threshold sensitivity

Run `retrack_threshold` with $\alpha \in \{0.20, 0.35, 0.50, 0.65, 0.80\}$ and collect the resulting height profiles. For each value compute the mean height difference from the 50%-threshold baseline and its along-track standard deviation. Plot all five profiles on the same axes and plot the mean difference against $\alpha$.

The slope of that relationship — metres of apparent height change per unit change in $\alpha$ — is the leading-edge sensitivity. Over a smooth interior with a clean leading edge it should be small (a few tens of centimetres from 20% to 80%). Over a marginal shot with a broad or multi-peaked waveform it will be much larger. Identify the two or three shots in your pass where leading-edge sensitivity exceeds the mean by more than a factor of three, and look at their raw waveforms to understand why.

### Task 2: slope-induced error estimate

A radar altimeter ranges to the nearest point on the surface, not the nadir point. On a slope of angle $\theta$ (small), the horizontal offset between the nearest point and nadir is approximately $R\sin\theta$ and the corresponding range error (shortest range minus nadir range) is

$$\Delta R \approx \frac{R^2 \tan^2\theta}{2R} = \frac{R\,\theta^2}{2} \quad (\theta \text{ in radians}),$$

which shifts the apparent surface upward by $\Delta h \approx R\,\theta^2 / 2$.

Use the along-track height gradient $dh/dx$ (from your retracked heights and the known 20-Hz shot spacing of about 350 m for CryoSat-2) as a proxy for the surface slope. For each shot estimate $\theta \approx |dh/dx|$ and compute $\Delta h$. Plot $\Delta h$ along the pass and identify where it exceeds 1 m. Those are the shots where the standard ESA L2 slope correction matters most, and where reprocessing with a digital elevation model to relocate the echo point {cite}`wingham2006` would improve the height.

In [ ]:
# --- Task 1: threshold sensitivity ---

alphas = [0.20, 0.35, 0.50, 0.65, 0.80]
heights_by_alpha = {}

for a in alphas:
    b_a = np.array([retrack_threshold(waveforms[i], alpha=a) for i in range(n_records)])
    h_a = alt_sat - (range_window_start + b_a * BIN_RANGE) - range_cor
    heights_by_alpha[a] = h_a

h_50 = heights_by_alpha[0.50]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for a in alphas:
    ax1.plot(lat, heights_by_alpha[a], lw=0.8, label=f'alpha={a}')
ax1.set_ylabel('height (m)')
ax1.set_title('Effect of threshold choice on retracked height')
ax1.legend(fontsize=8)

for a in [0.20, 0.35, 0.65, 0.80]:
    ax2.plot(lat, heights_by_alpha[a] - h_50, lw=0.8, label=f'alpha={a} − 0.50')
ax2.axhline(0, color='gray', lw=0.5)
ax2.set_ylabel('height difference from 50% (m)')
ax2.set_xlabel('latitude (°)')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('Mean (and std) of height difference from 50% threshold:')
for a in [0.20, 0.35, 0.65, 0.80]:
    d = heights_by_alpha[a] - h_50
    print(f'  alpha={a:.2f}:  mean {np.nanmean(d):+.3f} m,  std {np.nanstd(d):.3f} m')

# Along-track sensitivity per shot (range of height over all alpha values)
h_stack = np.stack([heights_by_alpha[a] for a in alphas], axis=0)
sensitivity = np.nanmax(h_stack, axis=0) - np.nanmin(h_stack, axis=0)
high_sens = np.where(sensitivity > 3 * np.nanmean(sensitivity))[0]
print(f'\nShots with sensitivity > 3x mean ({3*np.nanmean(sensitivity):.2f} m):')
for i in high_sens[:10]:
    print(f'  shot {i:4d}  lat {lat[i]:7.3f}°  sensitivity {sensitivity[i]:.2f} m')

In [ ]:
# --- Task 2: slope-induced range error ---

# Along-track shot spacing for CryoSat-2 SAR mode is roughly 300-350 m.
# Estimate it from the geodetic coordinates.
EARTH_RADIUS = 6_371_000.0   # m
dlat_rad = np.diff(np.deg2rad(lat))
dlon_rad = np.diff(np.deg2rad(lon))
dx = EARTH_RADIUS * np.sqrt(dlat_rad**2 + (dlon_rad * np.cos(np.deg2rad(lat[:-1])))**2)

# Surface slope: central difference, NaN at endpoints.
dh = np.gradient(h_thresh, edge_order=1)
dx_full = np.concatenate([[dx[0]], 0.5*(dx[:-1]+dx[1:]), [dx[-1]]])
slope = np.abs(dh / dx_full)   # dimensionless (radians for small angles)

R_typical = np.nanmean(R_thresh)
dh_slope = 0.5 * R_typical * slope**2   # metres, upward bias

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lat, dh_slope, lw=1.0, color='purple')
ax.axhline(1.0, color='gray', ls='--', lw=1, label='1 m level')
ax.set_xlabel('latitude (°)')
ax.set_ylabel('slope-induced height error (m)')
ax.set_title('Slope correction requirement along the pass')
ax.legend()
plt.tight_layout()
plt.show()

frac_over_1m = np.sum(dh_slope > 1.0) / n_records
print(f'Fraction of shots with slope error > 1 m: {100*frac_over_1m:.1f}%')
print(f'Maximum slope error: {np.nanmax(dh_slope):.2f} m')

## 7. Synthesis: why retracker consistency is the hidden variable in the elevation-change record

CryoSat-2 launched in 2010 and will eventually give way to another mission. ERS-1 flew in 1991. The elevation-change record that tells us how much ice the Greenland and Antarctic ice sheets have lost over the past three decades is a chain of overlapping missions, each with its own retracker, its own bandwidth, and its own antenna geometry. Joining those chains is not a matter of finding the same ice surface in each dataset — it is a matter of ensuring that each retracker was locating the *same part of the waveform* and therefore the same depth horizon in the firn.

The two retrackers you built make a useful mental model of the problem. Over an interior where dry-firn penetration is deep and seasonal, the OCOG and threshold-of-peak retrackers sit perhaps 0.5–2 m apart in height. If one mission used OCOG and the next used threshold-of-peak with a different $\alpha$, and if the transition happened during a year when firn properties were changing, the elevation record would carry an artificial step. Over the thirty-year record an undetected step of half a metre would bias the integrated mass loss estimate by an amount comparable to the interannual signal {cite}`wingham2006`.

This is why long-term ice-sheet mass balance ultimately rests on careful cross-calibration of retrackers across missions, on physical models of firn dielectric properties from {doc}`../foundations/snow-to-ice`, and on independent validation from airborne campaigns and GPS bedrock uplift. The signal the altimeter reports is real; getting it right requires understanding what the waveform is actually telling you — which is exactly what this lab has been practising.

**Questions for reflection**

1. Your Task 1 showed that changing $\alpha$ from 0.20 to 0.80 shifts the retracked height by some amount. If two missions used these two threshold values and produced overlapping data for one year, how would you estimate whether the height difference between them is due to the retracker or due to real elevation change? What additional data would you need?

2. The penetration depth of Ku-band microwaves increases as firn becomes drier and coarser-grained. If the Greenland interior experienced anomalously low summer melt for several consecutive years, what would you expect to see in the radar-altimeter elevation record — and would it reflect mass gain or a change in the scattering horizon? How does this connect to the combined use of radar and laser altimetry discussed at the end of {doc}`radar-altimetry`?

3. Task 2 shows that slope errors can exceed 1 m near an outlet glacier. The same outlet glacier is typically the region of fastest thinning — the place where mass loss is concentrated. Describe in one paragraph the chain of corrections and processing choices that would need to be right for a radar altimeter to measure the thinning rate of a fast-flowing outlet glacier to within 0.5 m per year, and identify the step you consider most uncertain.